# Lab 24: Causal ML — Double Machine Learning (Diagnostic Lab)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed code**. Your job:
1. Run the code
2. Identify what is wrong (you are told how many bugs, not where)
3. Fix the issue
4. Verify on a known DGP
5. Extend with Causal Forests

**Learning Objectives:**
- Implement manual 2-fold cross-fitting from scratch and debug common mistakes
- Understand why cross-fitting, treatment residualization, and the IV-style formula are each essential
- Estimate the ATE of 401(k) eligibility using the DoubleML package
- Assess robustness with sensitivity analysis
- Fit a Causal Forest (EconML) to estimate individual-level CATEs
- Compare subgroup DML to Causal Forest heterogeneity detection

**Verification checkpoints** are provided so you can confirm you found the right errors.

**Time estimate:** ~60 minutes

---

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages and import libraries
# -----------------------------------------------------------
!pip install -q doubleml econml

from doubleml import DoubleMLData, DoubleMLPLR
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load 401(k) data
data = fetch_401K(return_type='DataFrame')

print(f'Dataset shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
print('Libraries loaded. Ready to diagnose.')

---

## Part A: Manual Cross-Fitting — DIAGNOSE

The code below attempts to implement the DML algorithm manually using
2-fold cross-fitting. It has **three deliberate bugs**:

1. **Bug 1 (Data Leakage):** Uses the same data for training AND residual computation — violates cross-fitting
2. **Bug 2 (Missing Residualization):** Only residualizes the outcome $Y$, not the treatment $D$
3. **Bug 3 (Wrong Formula):** Uses `np.mean` of residual products instead of the correct IV-style formula for $\hat{\theta}$

**Your task:** Find all three bugs, explain why each matters, and fix them.

**The correct DML formula:**

$$\hat{\theta} = \frac{\sum_i \tilde{D}_i \tilde{Y}_i}{\sum_i \tilde{D}_i D_i}$$

where $\tilde{Y}_i = Y_i - \hat{\ell}(X_i)$ and $\tilde{D}_i = D_i - \hat{m}(X_i)$ are
the residuals from cross-fitted nuisance models.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has 3 bugs. Find and fix them all.
# Manual 2-fold cross-fitting DML
# -----------------------------------------------------------

# Generate simulated data with known ATE for verification
np.random.seed(42)
n = 5000
p = 100
TRUE_ATE = 5.0

X_sim = np.random.normal(0, 1, size=(n, p))
propensity = 1 / (1 + np.exp(-(0.5 * X_sim[:, 0] + 0.3 * X_sim[:, 1] + 0.2 * X_sim[:, 2])))
D_sim = np.random.binomial(1, propensity)
Y_sim = (TRUE_ATE * D_sim
         + 2.0 * X_sim[:, 0] + 1.5 * X_sim[:, 1] + 1.0 * X_sim[:, 2]
         + 0.5 * X_sim[:, 3] + 0.3 * X_sim[:, 4]
         + np.random.normal(0, 1, n))


def broken_dml(Y, D, X, random_state=42):
    """
    BROKEN manual DML implementation with 3 bugs.
    
    Bug 1: Uses same fold for training and prediction (no cross-fitting)
    Bug 2: Only residualizes Y, not D
    Bug 3: Uses np.mean(V_tilde * Y_tilde) instead of sum(V_tilde * Y_tilde) / sum(V_tilde * D)
    """
    n = len(Y)
    kf = KFold(n_splits=2, shuffle=True, random_state=random_state)
    
    Y_tilde = np.zeros(n)  # outcome residuals
    V_tilde = np.zeros(n)  # treatment residuals (but Bug 2 skips this)
    
    for train_idx, test_idx in kf.split(X):
        # --- Outcome model: Y ~ X ---
        ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        
        # BUG 1: Training and predicting on the SAME fold (train_idx)
        # Should train on train_idx, predict on test_idx
        ml_l.fit(X[train_idx], Y[train_idx])
        Y_hat = ml_l.predict(X[train_idx])       # <-- BUG: should be X[test_idx]
        Y_tilde[train_idx] = Y[train_idx] - Y_hat  # <-- BUG: should index test_idx
        
        # BUG 2: Missing treatment residualization entirely
        # Should fit ml_m on D ~ X and compute D_tilde = D - D_hat
        # Instead, just uses raw D as V_tilde
        V_tilde[train_idx] = D[train_idx]  # <-- BUG: should be D[test_idx] - D_hat[test_idx]
    
    # BUG 3: Wrong formula — uses np.mean instead of IV-style ratio
    # Correct: theta = sum(V_tilde * Y_tilde) / sum(V_tilde * D)
    theta = np.mean(V_tilde * Y_tilde)  # <-- BUG: wrong formula
    
    return theta


# Run the broken version
broken_ate = broken_dml(Y_sim, D_sim, X_sim)

print('=== BROKEN DML Results ===')
print(f'True ATE:    {TRUE_ATE:.2f}')
print(f'Broken ATE:  {broken_ate:.2f}')
print(f'Bias:        {broken_ate - TRUE_ATE:+.2f}')
print()
print('This estimate is far from the true ATE of 5.0.')
print('Find and fix the 3 bugs to recover the correct estimate.')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Fix the broken DML implementation
# -----------------------------------------------------------

def fixed_dml(Y, D, X, random_state=42):
    """
    FIXED manual DML: cross-fit both Y and D, use IV-style formula.
    """
    n = len(Y)
    kf = KFold(n_splits=2, shuffle=True, random_state=random_state)
    Y_tilde = np.zeros(n)
    V_tilde = np.zeros(n)

    for train_idx, test_idx in kf.split(X):
        # Fix 1: train on train_idx, predict on test_idx
        ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        ml_l.fit(X[train_idx], Y[train_idx])
        Y_tilde[test_idx] = Y[test_idx] - ml_l.predict(X[test_idx])

        # Fix 2: residualize D as well
        ml_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        ml_m.fit(X[train_idx], D[train_idx])
        V_tilde[test_idx] = D[test_idx] - ml_m.predict(X[test_idx])

    # Fix 3: correct IV-style formula
    theta = np.sum(V_tilde * Y_tilde) / np.sum(V_tilde * D)
    return theta


fixed_ate = fixed_dml(Y_sim, D_sim, X_sim)

print('=== FIXED DML Results ===')
print(f'True ATE:    {TRUE_ATE:.2f}')
print(f'Fixed ATE:   {fixed_ate:.2f}')
print(f'Bias:        {fixed_ate - TRUE_ATE:+.2f}')
print()
if abs(fixed_ate - TRUE_ATE) < 1.0:
    print('PASS — Fixed ATE is within 1.0 of the true value.')
else:
    print('FAIL — Fixed ATE is still far from 5.0. Check your fixes.')


---

## Part B: Package-Based DML

Now use the `doubleml` package to estimate the 401(k) ATE properly.
Less scaffolding than the 3916 lab — you should know the API from Part A.

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Estimate the 401(k) ATE with DoubleML
# -----------------------------------------------------------

y_col = 'net_tfa'
d_col = 'e401'
x_cols_dml = [c for c in data.columns if c not in [y_col, d_col]]

dml_data = DoubleMLData(data, y_col=y_col, d_cols=d_col, x_cols=x_cols_dml)

ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
ml_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)

dml_plr = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
dml_plr.fit()

print(dml_plr.summary)
print(f'\nATE: ${dml_plr.coef[0]:,.0f}')
print(f'95% CI: [{dml_plr.confint().iloc[0,0]:,.0f}, {dml_plr.confint().iloc[0,1]:,.0f}]')


In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Run sensitivity analysis
# -----------------------------------------------------------

dml_plr.sensitivity_analysis(cf_y=0.03, cf_d=0.03)
print(dml_plr.sensitivity_summary)
print()
print('Interpretation:')
print('The robustness value (rv) measures how strong an omitted confounder')
print('would need to be to drive the ATE to zero. rv > 1 means very robust;')
print('rv < 1 means a confounder of moderate strength could flip the sign.')
print('A positive rv means the 401(k) ATE estimate survives some confounding.')


---

## Part C: Causal Forests (EXTEND)

DML estimates a single ATE (or subgroup ATEs if you manually split).
**Causal Forests** from the `econml` package estimate individual-level
Conditional Average Treatment Effects (CATEs) — a treatment effect
for every observation.

The `CausalForestDML` combines:
- DML-style cross-fitting for debiasing
- Random Forest splitting to discover heterogeneity
- Honesty: separate samples for splitting and estimation

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Set up CausalForestDML from EconML
# -----------------------------------------------------------

# Prepare data arrays
y_col = 'net_tfa'
d_col = 'e401'
x_cols = [c for c in data.columns if c not in [y_col, d_col]]

Y = data[y_col].values
D = data[d_col].values.reshape(-1, 1)
X = data[x_cols].values

print(f'Y shape: {Y.shape}')
print(f'D shape: {D.shape}')
print(f'X shape: {X.shape}')
print(f'Covariates: {x_cols}')
print()

# Initialize CausalForestDML
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    model_t=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    n_estimators=500,   # Number of causal trees
    min_samples_leaf=20,
    max_depth=10,
    random_state=42,
    cv=5                # Cross-fitting folds
)

print('CausalForestDML configured.')
print('Next: fit the model and extract individual CATEs.')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Fit Causal Forest and extract CATE predictions
# -----------------------------------------------------------

# Step 1: Fit the Causal Forest
print('Fitting Causal Forest... (may take 1-3 minutes)')
cf.fit(Y, D, X=X)
print('Causal Forest fitted.')

# Step 2: Extract individual CATE predictions
cate_predictions = cf.effect(X).flatten()

# Step 3: Get 95% confidence intervals
cate_lower, cate_upper = cf.effect_interval(X, alpha=0.05)
cate_lower = cate_lower.flatten()
cate_upper = cate_upper.flatten()

print(f'CATE predictions shape: {cate_predictions.shape}')
print(f'Mean CATE:  ${np.mean(cate_predictions):,.0f}')
print(f'Std CATE:   ${np.std(cate_predictions):,.0f}')
print(f'Min CATE:   ${np.min(cate_predictions):,.0f}')
print(f'Max CATE:   ${np.max(cate_predictions):,.0f}')


In [ ]:
# -----------------------------------------------------------
# YOUR TASK — CATE histogram and high-response subgroup
# -----------------------------------------------------------

import os
os.makedirs('figures', exist_ok=True)

# Step 1: Plot histogram of individual CATE estimates
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cate_predictions, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(cate_predictions), color='red', linestyle='--', linewidth=2,
           label=f'Mean CATE: ${np.mean(cate_predictions):,.0f}')
ax.axvline(dml_plr.coef[0], color='orange', linestyle='--', linewidth=2,
           label=f'DML ATE: ${dml_plr.coef[0]:,.0f}')
ax.set_title('Distribution of Individual CATE Estimates (Causal Forest)', fontsize=13)
ax.set_xlabel('Estimated Treatment Effect ($)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('figures/cate_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 2: Identify high-response subgroup (CATE >= 75th percentile)
threshold = np.percentile(cate_predictions, 75)
high_mask = cate_predictions >= threshold
high_resp = data[high_mask].copy()
low_resp = data[~high_mask].copy()

print(f'High-response threshold (75th pct): ${threshold:,.0f}')
print(f'High-response n={len(high_resp)}, Low-response n={len(low_resp)}')
print()
compare_cols = ['inc', 'age', 'fsize', 'educ', 'pira', 'net_tfa']
comparison = pd.DataFrame({
    'High-Response': high_resp[compare_cols].mean(),
    'Low-Response': low_resp[compare_cols].mean()
})
print(comparison.round(1))


In [ ]:
# -----------------------------------------------------------
# EXTEND — Compare subgroup DML to Causal Forest CATE
# -----------------------------------------------------------

import os
os.makedirs('figures', exist_ok=True)

data['inc_quartile'] = pd.qcut(data['inc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
data['cate'] = cate_predictions

# Step 1: Mean Causal Forest CATE by income quartile
cate_by_q = data.groupby('inc_quartile', observed=True)['cate'].agg(['mean', 'std']).reset_index()
cate_by_q.columns = ['inc_quartile', 'mean_cate', 'std_cate']
print('Mean Causal Forest CATE by income quartile:')
print(cate_by_q.to_string(index=False))

# Step 2: Within-quartile variation
print('\nWithin-quartile CATE variation:')
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    mask = data['inc_quartile'].values == q
    std_q = cate_predictions[mask].std()
    mean_q = cate_predictions[mask].mean()
    print(f'  {q}: mean=${mean_q:,.0f}, std=${std_q:,.0f}')

# Step 3: Violin plot of CATE by income quartile
fig, ax = plt.subplots(figsize=(10, 6))
quartile_labels = ['Q1', 'Q2', 'Q3', 'Q4']
cate_groups = [cate_predictions[data['inc_quartile'].values == q] for q in quartile_labels]
ax.violinplot(cate_groups, positions=range(1, 5), showmeans=True)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(quartile_labels)
ax.set_xlabel('Income Quartile')
ax.set_ylabel('CATE ($)')
ax.set_title('Causal Forest CATE Distribution by Income Quartile', fontsize=13)
plt.tight_layout()
plt.savefig('figures/cate_by_quartile.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey question: Does the Causal Forest reveal heterogeneity')
print('WITHIN income quartiles that subgroup DML would miss?')
print('Answer: Yes — the large within-quartile std shows that income alone')
print('does not explain individual treatment effect variation.')


---

## Reflection

**When would you choose DML for ATE estimation vs. Causal Forests for CATE estimation?**

Choose DML when the research question requires a single credible average treatment effect — for example, when reporting a policy impact to regulators or comparing multiple interventions on a common scale. DML's ATE is easy to communicate and the robustness value from sensitivity analysis provides a concrete measure of how durable the estimate is to unmeasured confounding. Choose Causal Forests when the goal is targeting or personalization — identifying which subgroups respond most or least to treatment. CATEs require enough data per subgroup to be precise; with the 401(k) dataset (~9,000 observations), the Causal Forest reveals substantial within-income-quartile heterogeneity that coarse subgroup DML misses entirely. In practice, both methods are complementary: DML for the headline ATE with credible inference, Causal Forests for heterogeneity discovery and targeting analysis.


---

## Digital Portfolio: Institutional Signaling

### Generate Your Professional README

Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

```text
"I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and fixed a broken manual DML implementation (3 bugs:
  data leakage in cross-fitting, missing treatment residualization,
  wrong IV-formula for theta)
* Verified the fix recovers the true ATE (=5.0) on a simulated DGP
* Estimated the ATE of 401(k) eligibility on net financial assets
  using DoubleML with Random Forest nuisance learners and 5-fold cross-fitting
* Ran sensitivity analysis to assess robustness to unmeasured confounders
* Fit a CausalForestDML (EconML) to estimate individual-level CATEs
* Compared subgroup DML (quartile-level) to Causal Forest (individual-level)
  heterogeneity detection
* Key finding: [FILL IN — ATE, robustness, which method reveals finer heterogeneity?]

**Please write a README.md entry including:**
1. Project Title: Causal ML — DML and Causal Forests for Policy Evaluation
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it."
```

### Push to GitHub

```bash
cd econ-lab-24-causal-ml
git add notebooks/ figures/ README.md
git commit -m "Lab 24: Causal ML — DML & Causal Forests for 401(k) Policy"
git push origin main
```

Submit your GitHub repo link on Canvas.